<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex12.1-power-grid-stability-estimation/Ex12.1_00_system_check.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_12.1 · Notebook 00 — The System

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L12.1 (Power Grid Stability Estimation)**

## What Ex_12.1 is about

Recovering the state of a small transmission network from too few
measurements — first as an algebraic problem, then as a dynamic one — and
finishing by identifying the inertia of a machine from a disturbance. Most of
the system cannot be measured at all, and the physics is what fills the gap:
the power flow equations for the steady state, the swing equation for the
trajectory after a fault. Everything else follows from that.

There is no PDE and no rectangle here. The collocation points are buses and
instants, not points in a domain; `grad`, `to_tensor`, `MLP` and
`train_two_stage` are used exactly as everywhere else in Part 2.

### Goals

By the end of the exercise set you can

1. state what observability means as a rank condition on the measurement
   Jacobian, and say why it is a property of *where* the meters are rather than
   how many there are;
2. run weighted least squares as a baseline, read its normalised residuals, and
   identify a bad measurement from them;
3. add the power flow equations to the objective as a residual, and sweep the
   weight λ between measurements and physics — reading the trade-off off a
   curve, at the **unmetered** buses where it is visible;
4. represent a trajectory δ(t), ω(t) with a network, impose the swing equation
   by automatic differentiation, and build the initial condition into the trial
   solution instead of penalising it;
5. identify a physical parameter — machine inertia — as a trainable variable,
   and say from the fit alone whether the window you chose supports the answer;
6. report an estimate at an unmetered bus as what it is: an inference from the
   model, not a reading.

### Method — six notebooks, run in order

| notebook | what you do | problem / data | lecture |
|---|---|---|---|
| **00** | meet the network, solve its power flow, check the tools; nothing to write | the six-bus network | L12.1 |
| **01** | weighted least squares — the estimator utilities already run | the default and the thin measurement sets | L12.1 |
| **02** | the power flow equations as a residual; sweep λ | the thin set, where WLS cannot determine the state | L12.1 |
| **03** | simulate a fault cleared by a line trip, find its critical clearing time, then fit a network for δ(t), ω(t) with the swing equation imposed | 251 noisy frequency samples at the machine; δ is never measured | L12.1 |
| **04** | the inverse problem: recover inertia H and damping D | a frequency trace and the window you choose | L12.1 |
| **05** | compare the estimators and write the report | the results saved in `Ex12.1_outputs/` | — |

Later notebooks load results saved by earlier ones, so run them in order.
Notebooks 01 to 04 also come in lighter `_light` variants.

### Applications

* **State estimation in a control room.** A DK2-representative network: bus 0
  is the connection to the Nordic synchronous area through Sweden, bus 1 local
  generation, buses 2–4 load, and bus 5 the HVDC link to Germany. Real Danish
  injections (Energinet) and real Nordic frequency (Fingrid) are used; the line
  impedances are plausible textbook values, not measured ones.
* **Inertia monitoring.** As synchronous plant is replaced by
  converter-connected generation, the effective inertia of a region falls,
  varies through the day and is not directly metered. Notebook 04 shows why it
  can only be identified from a disturbance.
* **Reporting what is supported.** Every table separates metered from
  unmetered buses and gives the worst bus alongside the mean. The last report
  question asks which of your numbers you would show a control-room operator.

## What this notebook does

**Read and run. You are not asked to change anything here.**

This notebook introduces the network you will spend the exercise estimating,
solves its power flow, and checks that everything the later notebooks need is
present and working. Run it first — the notebooks are sequential, and this one
saves the reference state that notebooks 01 to 04 load.

### What you are looking at

Six buses. Bus 0 is the connection to the Nordic synchronous area through
Sweden and behaves as an infinite bus. Bus 1 is local generation. Buses 2, 3
and 4 are load. Bus 5 is the HVDC link to Germany, which enters as a fixed
injection rather than as a synchronous branch — a DC link transfers power but
not synchronism, so it is a boundary condition, not a branch.

### The honest bit, stated once

The network is **DK2-representative, not DK2**. Its structure follows eastern
Denmark; the line impedances are plausible textbook values, *not* measured
ones. No TSO publishes a nodal model with impedances. Real Danish load and real
Nordic frequency are used later; the topology is not real, and any conclusion
that depends on a specific reactance inherits that.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex12.1-power-grid-stability-estimation/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.
import course_core as cc
import problem as pb
pb.RESULTS = cc.keep_outputs("Ex12.1_outputs")


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · Tools, checked before you rely on them

Two of the three modules beside this notebook are shared with every other
exercise in Part 2. The third, `problem.py`, is this network — the buses, the
branches, the power flow, the machines, the Danish data and the plots.

| | |
|---|---|
| `course_core.py` | shared by the whole course — `set_seed`, `MLP`, `to_tensor`, `check` |
| `pinn_core.py` | the PDE machinery — `grad`, `d2`, samplers, `train_two_stage` |
| `problem.py` | **this** problem — network, power flow, dynamics, data, plots |

One thing to notice before you go looking for it. `pinn_core`'s samplers —
`interior_points`, `boundary_points`, `grid_points` — sample a **rectangle**,
and this exercise has no rectangle in it. The algebraic problem lives on six
buses and the dynamic one on a time axis, so the collocation points here are
buses and instants. That is not a gap in the tooling; it is what a graph
problem looks like. What does carry over unchanged is `grad`, and the cell
below is the check that it works.

In [ ]:
import sys, matplotlib

print("python     ", sys.version.split()[0])
print("numpy      ", np.__version__)
print("matplotlib ", matplotlib.__version__)
print("torch      ", torch.__version__)
print("cuda        available:", torch.cuda.is_available(), " (not needed)")

# Automatic differentiation, against something you can differentiate by hand.
# Notebooks 03 and 04 take d(delta)/dt and d(omega)/dt of a network exactly
# this way, so if this fails nothing after notebook 02 can work.
#   f(t) = sin(3t)   ->   df/dt = 3 cos(3t)
t = to_tensor(np.linspace(0.0, 1.0, 64).reshape(-1, 1), requires_grad=True)
f = torch.sin(3 * t)

print()
check("df/dt = 3 cos(3t)", to_numpy(grad(f, t)),
      3 * np.cos(3 * to_numpy(t)), tol=1e-9)
check_shape("grad returns one column per input", to_numpy(grad(f, t)), (64, 1))

**Expected output**

> Four version numbers, `device: cpu` and `dtype: torch.float64`, then two
> `PASS` lines with an error at machine precision.
>
> Double precision is deliberate and it is not the deep-learning default. State
> estimation is ill-conditioned — the normal equations in notebook 01 are formed
> from a Jacobian whose columns differ in scale by orders of magnitude — and in
> float32 the answer would be dominated by rounding.
>
> `to_tensor(..., requires_grad=True)` is what makes a coordinate
> differentiable. Omit it and `grad` returns `None`, which is the single most
> common first error in Part 2.

---

## 2 · The network

`pb.build_ybus()` assembles the bus admittance matrix from the branch list. $Y$
is the graph: it holds the topology and the line parameters in one sparse,
symmetric object, and every equation in this exercise is written against it.

The injections that follow from a state $(V,\theta)$ are

$$P_i = V_i\sum_k V_k\left(G_{ik}\cos\theta_{ik}+B_{ik}\sin\theta_{ik}\right),
\qquad
Q_i = V_i\sum_k V_k\left(G_{ik}\sin\theta_{ik}-B_{ik}\cos\theta_{ik}\right)$$

with $\theta_{ik}=\theta_i-\theta_k$. That pair is the whole physical content
of the algebraic half of this exercise.

In [ ]:
Y = pb.build_ybus()
P, Q = pb.injections()

print("buses:")
for i, (name, kind, p, q) in enumerate(pb.BUSES):
    print(f"  {i}  {name:<18s} {kind:<6s}  P = {p:+.2f}   Q = {q:+.2f}  p.u.")

print("\nbranches:")
for k, (f_, t_, r, x, b) in enumerate(pb.BRANCHES):
    print(f"  {k}  {f_} - {t_}   r = {r:.4f}   x = {x:.4f}")

print(f"\nY is {Y.shape[0]}x{Y.shape[1]}, "
      f"{np.count_nonzero(Y)} non-zero entries out of {Y.size}")

**Expected output**

> Six buses listed, bus 0 as `slack`, bus 1 as `gen`, the rest `load`.
> Six branches. Y is 6x6 with **18 non-zero entries** — six on the diagonal plus
> two per branch. A dense Y here would mean you built it wrong.

## 3 · The power flow

Newton-Raphson on the injections. This is the *reference* state: the true
answer that the estimators in later notebooks are trying to recover from
noisy, incomplete measurements. In a control room you never have this.

Note what a healthy solution looks like — voltages in a narrow band just below
1 p.u., and small angle differences. A converged solution with voltages at 0.8
p.u. is telling you the network is overloaded, not that the solver worked.

In [ ]:
V, th, converged, iters = pb.solve_power_flow(Y, P, Q)

print(f"converged: {converged}   in {iters} iterations\n")
print(f"  {'bus':>4}{'|V| (p.u.)':>13}{'angle (deg)':>14}")
for i in range(pb.N_BUS):
    print(f"  {i:>4}{V[i]:>13.4f}{np.degrees(th[i]):>14.3f}")

Pc, Qc = pb.pq_from_state(V, th, Y)
print(f"\nlargest injection mismatch (non-slack): "
      f"{np.max(np.abs(Pc[1:] - P[1:])):.2e}")

**Expected output**

> `converged: True   in 5 iterations`
>
> Voltages between **0.9706 and 1.0000** p.u., angles within about 2.5 degrees
> of zero. The largest mismatch should be around **1e-15** — machine precision. That is
> the solver telling you it really solved the equations, not merely stopped
> iterating.

## 4 · What is measured, and what is not

A `pb.MeasurementSet` says which quantities exist and how accurate they are.
Two sets are provided:

* **default** — PMUs at two buses, SCADA injections at five. Observable.
* **thin** — one PMU, two injections. *Not* observable.

Observability is a rank condition on the measurement Jacobian, and it is a
property of *where* the meters are, not how many there are. If the rank is
short, some combination of the state is not determined by the data at all —
and no estimator, however clever, recovers it from measurements alone.

That second set is where this whole exercise lives.

In [ ]:
for label, ms in [("default", pb.default_measurements()),
                  ("thin", pb.thin_measurements())]:
    ok, rank, need = pb.observable(ms, Y)
    print(f"{label:>8}: {len(ms):2d} measurements   "
          f"observable = {str(ok):5s}   rank {rank}/{need}")
    print(f"          metered buses   {ms.measured_buses()}")
    print(f"          unmetered buses {ms.unmeasured_buses()}")

**Expected output**

> ```
>  default: 14 measurements   observable = True    rank 11/11
>           metered buses   [0, 1, 2, 3, 4]
>           unmetered buses [5]
>     thin:  6 measurements   observable = False   rank 5/11
>           metered buses   [0, 3]
>           unmetered buses [1, 2, 4, 5]
> ```
>
> Rank 5 out of 11 needed. Six of the eleven unknowns are simply not determined.
>
> Eleven, not twelve: the slack bus fixes the angle reference, so five angles are
> unknown, but its voltage *magnitude* is estimated like any other, so all six
> magnitudes are. Angles and magnitudes do not exclude the same bus.

## 5 · Real Danish load

The operating point above is the network's design loading. Here we bring in the
*shape* of a real DK2 day from Energinet's open API — no key needed — and map it
onto the network.

Only the shape transfers. Dropping DK2's actual megawatts onto a six-bus case
designed to carry a fraction of them diverges the power flow, which is the
first thing everyone tries. `pb.scale_to_network` maps the peak hour onto the
network's design loading and reports the implied MVA base, which is the honest
statement of what the per-unit numbers mean.

If you have no network access the loader falls back to a synthetic curve and
says so. A result computed on synthetic data is not a result about Denmark.

In [ ]:
pb.use_course_style()

day = pb.dk2_load()
print(f"  source: {day['source']}")
print(f"  peak {day['load_mw'].max():.0f} MW at hour {int(np.argmax(day['load_mw']))}, "
      f"trough {day['load_mw'].min():.0f} MW")

fig, ax = plt.subplots(figsize=(6.4, 2.8))
ax.plot(day["hours"], day["load_mw"], "o-", color=pb.CYAN, ms=3)
ax.set_xlabel("hour"); ax.set_ylabel("DK2 load (MW)")
ax.set_title(f"source: {day['source']}", fontsize=10)
plt.show()

for h in (3, 9, 18):
    Ph, hour, base = pb.scale_to_network(day["load_mw"], P, hour=h)
    Vh, thh, okh, _ = pb.solve_power_flow(Y, Ph, Q)
    print(f"  hour {h:2d}:  load {-Ph[Ph<0].sum():.3f} p.u.   "
          f"implied base {base:.0f} MVA   converged {okh}   Vmin {Vh.min():.4f}")

**Expected output**

> A load curve with a night trough, a morning ramp and an evening peak.
>
> All three hours converge, with the implied base around **740 MVA** and minimum
> voltages near 0.97. If an hour fails to converge, the scaling is wrong — check
> you passed the *shape*, not the megawatts.

## 6 · Save the reference state

Later notebooks load this. If the file is missing there — a fresh Colab
runtime, for instance, since every Colab tab is a separate machine with its
own filesystem — `pb.load("00_reference")` rebuilds it on the spot and says
so, because it is exactly the computation this notebook just performed.
The training results of notebooks 01–04 are *not* rebuilt that way: those
are your runs, and notebook 05 compares them. To carry them across runtimes,
download the `Ex12.1_outputs/` folder and upload it in the next tab.


In [ ]:
pb.save("00_reference", V=V, th=th, P=P, Q=Q,
        load_mw=day["load_mw"], source=np.array([day["source"]]))
print("\nnotebook 00 complete — go to 01_wls_baseline")

**Expected output**

> `saved -> Ex12.1_outputs/00_reference.npz  (V, th, P, Q, load_mw, source)`

### This solver is for clarity, not for scale

The Newton-Raphson solver here is written out in full, deliberately, so that
the load flow is not a black box. Two things about it would be unacceptable in
a production tool, and both are worth knowing before you take this approach to
a larger system.

The first is the **numerical Jacobian**. Each entry is estimated by perturbing
one state variable and re-evaluating the power injections, so one Jacobian
costs of the order of 2n full evaluations. An analytic Jacobian, which the real
tools use, costs one pass and is exact.

The second is **dense linear algebra**. The admittance matrix of a real network
is extremely sparse, because a bus connects to a handful of neighbours rather
than to all the others. Storing and factorising it densely scales as the cube
of the bus count, which is why this implementation is comfortable on six buses
and impractical beyond a few dozen.

There is also no PV bus handling with reactive limits, which every real network
needs and which changes the structure of the problem when a generator hits its
limit.

**If you go on to real-scale work**, the tools are
[pandapower](https://pandapower.readthedocs.io) in Python and
[PowerSystems.jl](https://nrel-sienna.github.io/PowerSystems.jl) in Julia. Both
use analytic Jacobians, sparse storage and proper bus-type handling, and both
ship the standard IEEE test cases including the 118-bus system. Naming them
here is meant to save you the discovery step, not to suggest that writing this
solver was wasted: reading one of those libraries is far easier once you have
written the thing it replaces.


---

Continue with **[`Ex12.1_01_wls_baseline.ipynb`](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex12.1-power-grid-stability-estimation/Ex12.1_01_wls_baseline.ipynb)**.
